# IMPORT LIBRARY

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

# CONNECTION TO DATABASE

In [ ]:
load_dotenv("../.env")

conn = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connection created")

# AGENT SALES ANALYSIS

##
RETRIVE DATA

In [ ]:
dataset = '''SELECT 
                 a.agent_id
                ,a.agent_city
                ,a.experience_years
                ,a.base_commission_rate
                ,a.agent_rating 
                ,l.listing_id
                ,l.listing_date
                ,l.listing_status
                ,l.close_date
                ,t.transaction_id
                ,t.city
                ,t.deal_price
                ,t.deal_date
                ,t.deal_status
                ,t.commission_amount
                ,p.city AS city_prop
                ,p.property_type
                ,c.home_city
            FROM listings l
            LEFT JOIN transactions t ON l.listing_id = t.listing_id
            LEFT JOIN agents a ON a.agent_id = l.agent_id
            LEFT JOIN properties p ON p.property_id = l.property_id
            LEFT JOIN customers c ON c.customer_id = t.customer_id
            '''

df_dataset = pd.read_sql(dataset,conn)

df_dataset = df_dataset.sort_values('agent_id').reset_index(drop = True)
len(df_dataset)

In [ ]:
df_dataset['deal_date'] = pd.to_datetime(df_dataset['deal_date'])
df_dataset['listing_date'] = pd.to_datetime(df_dataset['listing_date'])
df_dataset['close_date'] = pd.to_datetime(df_dataset['close_date'])

df_dataset['days_on_market'] = np.where(\
    df_dataset['close_date'].isna(),
    (pd.Timestamp('2024-12-31') - df_dataset['listing_date']).dt.days,
    (df_dataset['close_date'] - df_dataset['listing_date']).dt.days)


In [ ]:
df_agent = (\
    df_dataset
    .groupby(['agent_id', 'agent_city', 
              'experience_years', 'base_commission_rate',
              'agent_rating'])
       .agg(total_listing = ('listing_id','count'))
       .reset_index()
)

df_completed =(\
    df_dataset[
        ((df_dataset['close_date'].dt.year)<=2024) & 
        (df_dataset['deal_status']=='Completed')
        ]
)

df_active_listing = (\
    df_dataset[
        (df_dataset['listing_status']=='Active')|
        ((df_dataset['close_date'].dt.year)>=2025)]
)



## 
BASELINE AGENT FINANCIAL & EARNINGS

###
1) Agent Overview

In [ ]:
df_completed_agent = (\
    df_completed
    .groupby('agent_id')
    .agg(total_completed_transactions = ('transaction_id','count'),
        total_revenue = ('deal_price','sum'),
        total_commission = ('commission_amount','sum'),
        median_DOM = ('days_on_market','median'))
    .reset_index()
)

df_completed_agent['median_DOM'] = df_completed_agent['median_DOM'].astype(int)
df_agent = df_agent.merge(df_completed_agent, on='agent_id')


df_agent['contribution_share'] = (\
    df_agent['total_revenue']/
    (df_agent['total_revenue'].sum())
    *100)


df_active_listing_agent = (\
    df_active_listing
    .groupby('agent_id')
    .agg(total_active_listing = ('listing_id','count'))
    .reset_index()
)
df_agent = df_agent.merge(df_active_listing_agent,on = 'agent_id', how ='left')


df_agent['closing_rate'] = (\
    df_agent['total_completed_transactions']/
    df_agent['total_listing']
)

In [ ]:
print(f'Number of Agent = {len(df_agent)}')
print(f"Avg Listing per Agent = {df_agent['total_listing'].mean():.0f} Listings")
print(f"Avg total transacton per Agent = {df_agent['total_completed_transactions'].mean():.0f} Transactions")
print(f"Median Revenue per Agent = {df_agent['total_revenue'].median()/1e6:.2f} M")
print(f"AVG Revenue per Agent = {df_agent['total_revenue'].mean()/1e6:.2f} M")
print(f"Median Commision per Agent = {df_agent['total_commission'].median()/1e6:.2f} M")
print(f"AVG Commision per Agent = {df_agent['total_commission'].mean()/1e6:.2f} M")

###
3) Agent Eranings Resume

In [ ]:
df_agent_rev = (
    df_agent[['agent_id','total_revenue','total_commission','contribution_share','total_completed_transactions']]
              .sort_values('total_revenue',ascending = False).reset_index(drop = True)
)
df_agent_rev

##
AGENT PRODUCTIVITY & SALES VELOCITY

In [ ]:
df_agent_productivity = (\
    df_agent[['agent_id','closing_rate','median_DOM','total_listing','total_active_listing']]
    .sort_values('median_DOM').reset_index(drop=True)
)

display(df_agent_productivity)
print(f"AVG DOM Agent :{round(df_agent_productivity['median_DOM'].mean(),0)}")
print(f"AVG total listing activer :{round(df_agent_productivity['total_active_listing'].mean(),0)}")

##
AGENT QUALITY & PERFORMANCE PREDICTORS

###
1) Sales Performance by Agent

In [ ]:
df_agent_quality =(\
    df_agent[['agent_id','agent_rating','experience_years','total_completed_transactions']]
    .sort_values('agent_rating',ascending = False)
    .reset_index(drop = True)
)

df_agent_quality

###
3) Agent Classification Based on Years of Experience

In [ ]:

df_agent['status'] = np.where(
    df_agent['experience_years'] < 2, 
    'Junior', 
    np.where(
        (df_agent['experience_years'] >= 2) & (df_agent['experience_years'] < 5), 
        'Mid', 
        'Senior'
    )
)

df_agent['status'].value_counts()

###
4) Agent Classification Based on Revenue Generated

In [ ]:
df_agent_sales = df_completed.groupby('agent_id')['deal_price'].sum().reset_index()

df_agent_sales['quartile'] = pd.qcut(
    df_agent_sales['deal_price'], 
    q=4, 
    labels=['Q1 (Bottom)', 'Q2', 'Q3', 'Q4 (Top)']
)

df_agent_sales['performance_tier'] = df_agent_sales['quartile'].apply(
    lambda x: 'Top Performer' if x == 'Q4 (Top)' else ('Bottom Performer' if x == 'Q1 (Bottom)' else 'Mid Performer')
)

df_agent_sales['performance_tier'].value_counts()

In [ ]:
df_completed.groupby(['agent_id', 'city']).size().reset_index(name='total_deals')

###
5) Closing Drivers by Agent

In [ ]:
correlation = df_agent_quality[['agent_rating','experience_years','total_completed_transactions']].corr(method='spearman')
correlation

## 
AGENT PERFORMANCE BY CITY

###
1) Agent Sales Performance by Agent City

In [ ]:
data =df_completed.groupby('agent_city').agg(total_transaction = ('transaction_id','count'),
                                              total_agent = ('agent_id','nunique'),
                                              avg_DOM = ('days_on_market','mean')).sort_values('avg_DOM').reset_index()

data['avg_trans_per_agent'] = round(data['total_transaction']/data['total_agent'],0)
data['avg_DOM'] = data['avg_DOM'].round(0)

data

##
2) Agent Transaction Distribution by City

In [ ]:
transaction_agent_distibution = pd.crosstab(
    index=df_completed['agent_city'],
    columns=df_completed['city_prop'],
    values=df_completed['transaction_id'],
    aggfunc='count'
)

display(transaction_agent_distibution)


###
3) Agent Quality Based on Conversion Rate %

In [ ]:

#Calculating Convertion Rate for Total Listing by Agent_City
data1 = (\
    df_dataset
    .groupby('agent_city')
    ['listing_id'].count()
    .reset_index(name ='total_listing_by_agent_city')
)
data2 = (\
    df_completed
    .groupby('agent_city')
    ['transaction_id'].count()
    .reset_index(name = 'total_transaction_by_agent_city')
)
conversion_rate = (\
    data1
    .merge(
        data2, 
        on ='agent_city')
)
conversion_rate['conversion_rate'] = (\
    round(
    conversion_rate['total_transaction_by_agent_city']/
    conversion_rate['total_listing_by_agent_city']
    *100,0)
)


#Calculating Convertion Rate a Cross City for Total Listing by Agent_City
data4= (\
    df_completed[
        df_completed['city_prop']!=df_completed['agent_city']
        ]
    .groupby('agent_city')
    .agg(trans_cross_city = ('transaction_id','count'))
)

data5= (\
    df_dataset[
        df_dataset['agent_city']!=df_dataset['city_prop']
        ]
    .groupby('agent_city')
    .agg(listing_cross_city = ('listing_id','count'))
)

conversion_rate_cross_city = (\
    data4
    .merge(
        data5, on = 'agent_city')
).reset_index()

conversion_rate_cross_city['conversion_rate'] = (\
    round(
    conversion_rate_cross_city['trans_cross_city']/
    conversion_rate_cross_city['listing_cross_city']
    *100,0)
)


#Calculating Convertion Rate Local for Total Listing by Agent_City
data6= (\
    df_completed[df_completed['city_prop']==df_completed['agent_city']]
    .groupby('agent_city')
    .agg(trans_local= ('transaction_id','count'))
)

data7= (\
    df_dataset[df_dataset['agent_city']==df_dataset['city_prop']]
    .groupby('agent_city')
    .agg(listing_local = ('listing_id','count'))
)

conversion_rate_local = (\
    data6
    .merge(
        data7, 
        on = 'agent_city')
).reset_index()

conversion_rate_local['conversion_rate'] =(\
    round(
    conversion_rate_local['trans_local']/
    conversion_rate_local['listing_local']
    *100,0)
)


display(conversion_rate)
display(conversion_rate_cross_city)
display(conversion_rate_local)

###
4) Agent Distribution by Deal City

In [ ]:
sns.set_style(style='whitegrid')
fig, ax1 = plt.subplots(figsize=(10, 6))

x = np.arange(len(conversion_rate_local["agent_city"]))
width = 0.4

rects1 = ax1.bar(
    x - width/2,
    conversion_rate_local["trans_local"],
    width,
    color="#CDE6FF",
    label="Total Local Transaction",
)

rects2 = ax1.bar(
    x + width/2,
    conversion_rate_cross_city["trans_cross_city"],
    width,
    color="#1763A1",
    label="Total Transaction Cross City",
)

ax1.set_ylabel("Total Transaction", fontsize=11, fontweight="bold")
ax1.set_xlabel("Agent City", fontsize=11, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(conversion_rate_local["agent_city"].astype(str), fontsize=11)
ax1.grid(axis="y", linestyle="--", alpha=0.5)

ax1.bar_label(rects1, padding=3, fmt='%d')
ax1.bar_label(rects2, padding=3, fmt='%d')

plt.title(
    "Local vs Cross-City Transactions by Agent City",
    fontsize=13,
    fontweight="bold",
    pad=20,
    loc="center",
)

bar1, labels1 = ax1.get_legend_handles_labels()
ax1.legend(
    bar1,
    labels1,
    bbox_to_anchor=(0.5, -0.15),
    loc="upper center",
    ncol = 2,
    frameon=True,
    facecolor="white",
    edgecolor="none",
)

sns.despine(top=True, right=False)
plt.tight_layout()
plt.show()

###
5) Agent Quality Based on Market Share(%)

In [ ]:
trans_local= (\
    df_completed[df_completed['city_prop']==df_completed['agent_city']]
    .groupby('agent_city')
    .agg(trans_local= ('transaction_id','count'))
)

listing_by_city = (\
    df_completed
    .groupby('city_prop')
    .agg(total_trans_by_city = ('transaction_id','count'))
)

market_share_by_listing_city = trans_local.merge(
    listing_by_city, 
    left_index=True, 
    right_index=True, 
    how='left'
).reset_index()

market_share_by_listing_city['market_share%'] = (\
    round(market_share_by_listing_city['trans_local']/
          market_share_by_listing_city['total_trans_by_city']
          *100.2)
)

market_share_by_listing_city

In [ ]:
display(pd.crosstab(index=df_completed['property_type'],
            columns = df_completed['home_city']))

display(pd.crosstab(index=df_completed['city_prop'],
            columns=df_completed['home_city']))

display(pd.crosstab(index=df_completed['agent_city'],
                               columns = df_completed['home_city']))